## Create SQL Functions for Genie Agents

### Function 1 - Get emissions for a facility

The sustainability manager can ask Genie to retrieve emissions for a particular facility and a year

In [ ]:
%sql

CREATE OR REPLACE FUNCTION carbonops.gold.get_facility_emissions(
    p_facility STRING,
    p_year INT
)
RETURNS TABLE (
    facility_name STRING,
    reporting_year INT,
    scope1_emissions DOUBLE,
    scope2_emissions DOUBLE,
    total_emissions DOUBLE
)
RETURN
    SELECT
        facility_name,
        year AS reporting_year,
        CAST(SUM(scope1_tco2e) AS DOUBLE),
        CAST(SUM(scope2_tco2e) AS DOUBLE),
        CAST(SUM(total_tco2e) AS DOUBLE)
    FROM carbonops.gold.facility_emissions
    WHERE LOWER(facility_name) = LOWER(p_facility)
      AND year = p_year
    GROUP BY facility_name, year;

In [ ]:
%sql

SELECT *
FROM carbonops.gold.get_facility_emissions(
    'Mumbai Automotive Plant',
    2025
);

### Function 2 - Get facilities exceeding their emission targets

In [ ]:
%sql

CREATE OR REPLACE FUNCTION carbonops.gold.get_target_breaches(
    p_year INT
)
RETURNS TABLE (
    facility_name STRING,
    reporting_year INT,
    annual_target DOUBLE,
    actual_emissions DOUBLE,
    utilization_pct DOUBLE
)
RETURN
    WITH latest AS (
        SELECT
            facility_name,
            facility_id,
            year,
            annual_target_tco2e,
            ytd_emissions_tco2e,
            target_utilization_pct,
            ROW_NUMBER() OVER (
                PARTITION BY facility_id, year
                ORDER BY period_end DESC
            ) AS rn
        FROM carbonops.gold.target_performance
        WHERE year = p_year
    )
    SELECT
        facility_name,
        year,
        CAST(annual_target_tco2e AS DOUBLE),
        CAST(ytd_emissions_tco2e AS DOUBLE),
        CAST(target_utilization_pct AS DOUBLE)
    FROM latest
    WHERE rn = 1
      AND annual_target_tco2e IS NOT NULL
      AND ytd_emissions_tco2e > annual_target_tco2e;

In [ ]:
%sql

SELECT *
FROM carbonops.gold.get_target_breaches(2025);